# Ray Paths With `3D_Reference_Model_Margarit`

This notebook reuses the selected-event logic from the ray-path workflow, but computes only `P` travel times and ray paths in a spherical `pykonal` model where:

- the lunar surface radius is constant,
- crustal thickness and Moho depth vary laterally from the Kim et al. grid,
- the crustal `Vp/Vs/rho` structure is stretched from the 1D Garcia-style template,
- the mantle below the local Moho remains 1D.


In [2]:
import sys
print(sys.executable)


/Users/ramonmargarit/IPGP Dropbox/Ramon Margarit/PhD/Science/Modelling_Envelopes/.venv-raypaths/bin/python


In [ ]:
# If needed, install the packages in this kernel first:
# %pip install numpy pandas matplotlib xarray openpyxl pykonal

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
from IPython.display import display

try:
    import pykonal
except ModuleNotFoundError:
    pykonal = None
    print("`pykonal` is not installed in this kernel. Install it with `%pip install pykonal`.")

RMOON_KM = 1737.4
PHASE_NAME = "P"
SOURCE_DEPTH_KM = 30.0

GRAIL_GRID = Path("/Users/ramonmargarit/IPGP Dropbox/Ramon Margarit/PhD/Science/Modelling_Envelopes/data/raw/thick-2-35.5-12-3400.txt")
MODEL_FILE = Path("/Users/ramonmargarit/Desktop/vpremoon_mantle_closed.tvel")
SELECTED_EVENTS_CSV = Path(
    "/Users/ramonmargarit/Phd/Projects/Heterogeneities_Mantle/Heterogeneities-project/data/processed/selected_events/selected_events_best_7_bands_fixed_hold0_LOW0p75_MINPOST4_KNEG0_KPREPOS0.csv"
)
XLSX = Path("/Users/ramonmargarit/IPGP Dropbox/Ramon Margarit/PhD/Science/Modelling_Envelopes/data/processed/Shallow_processed_RESULTS.xlsx")
SHEET = "best_7_bands_fixed_hold0"
RESULTS_CSV = Path(
    "/Users/ramonmargarit/IPGP Dropbox/Ramon Margarit/PhD/Science/Modelling_Envelopes/notebooks/results/ray_paths_3d_reference_model_margarit.csv"
)

FC = 5.0
BANDS = np.array([3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0])
SCENARIO = dict(LOWER_TOL=0.75, MIN_POST=4, K_NEG=0, K_PRE_POS=0)

APOLLO_STATIONS = {
    "S16": {"lat": -8.9730, "lon": 15.5000},
    "S14": {"lat": -3.6440, "lon": -17.4775},
    "S15": {"lat": 26.1322, "lon": 3.6339},
}

MAX_EVENTS = None

NR = 45
NTHETA = 61
NPHI = 121

TRACE_METHOD = "native"
STRICT_NATIVE_TRACE = False
TRACE_STEP_KM = 2.0
RECEIVER_DEPTH_KM = 1.0


`pykonal` is not installed in this kernel. Install it with `%pip install pykonal`.


In [4]:
def _first_existing_col(cols, candidates):
    for col in candidates:
        if col in cols:
            return col
    return None


def _match_longitude(lon_values, query_lon_deg):
    lon_values = np.asarray(lon_values, dtype=float)
    if np.nanmin(lon_values) < 0:
        return ((query_lon_deg + 180.0) % 360.0) - 180.0
    return np.mod(query_lon_deg, 360.0)


def load_grail_crust(path):
    if not path.exists():
        raise FileNotFoundError(f"GRAIL crust file not found: {path}")

    grid_m = np.loadtxt(path)
    if grid_m.shape != (1803, 3605):
        raise ValueError(f"Unexpected Kim-grid shape: {grid_m.shape}. Expected (1803, 3605).")

    lat = np.linspace(90.0, -90.0, grid_m.shape[0])
    lon = np.linspace(0.0, 360.0, grid_m.shape[1])
    crust_km = xr.DataArray(
        grid_m / 1e3,
        coords={"lat": lat, "lon": lon},
        dims=("lat", "lon"),
        name="crust_thickness_km",
        attrs={"units": "km", "source": str(path)},
    )
    return crust_km.sortby("lat").sortby("lon")


def read_vpremoon_model(model_file):
    rows = []
    started = False

    with open(model_file, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            parts = line.split()
            if not started:
                if len(parts) >= 4 and parts[0].lower() == "depth_km":
                    started = True
                continue

            if len(parts) < 4:
                continue

            try:
                rows.append([float(parts[0]), float(parts[1]), float(parts[2]), float(parts[3])])
            except ValueError:
                continue

    if not rows:
        raise ValueError(f"No model rows found in file: {model_file}")

    model_df = pd.DataFrame(rows, columns=["depth_km", "vp_km_s", "vs_km_s", "rho_g_cm3"])
    model_df["radius_km"] = RMOON_KM - model_df["depth_km"]
    return model_df


def collapse_depth_profile(model_df, keep):
    return model_df.sort_values("depth_km").drop_duplicates(subset="depth_km", keep=keep).reset_index(drop=True)


def interpolate_profile(depths_km, profile_df, column):
    x = profile_df["depth_km"].to_numpy(dtype=float)
    y = profile_df[column].to_numpy(dtype=float)
    return np.interp(np.asarray(depths_km, dtype=float), x, y, left=y[0], right=y[-1])


def sample_crust_thickness(grail_crust_km, lat_deg, lon_deg):
    crust_lon = _match_longitude(grail_crust_km["lon"].values, lon_deg)
    return float(grail_crust_km.interp(lat=lat_deg, lon=crust_lon, kwargs={"fill_value": np.nan}).item())


def build_reference_model_margarit_3d(grail_crust_km, radial_model_df):
    reference_crust_thickness_km = float(
        radial_model_df.loc[
            radial_model_df["depth_km"].duplicated(keep=False) & (radial_model_df["depth_km"] <= 100.0),
            "depth_km",
        ].max()
    )

    crust_profile_df = collapse_depth_profile(
        radial_model_df.loc[radial_model_df["depth_km"] <= reference_crust_thickness_km], keep="first"
    )
    mantle_profile_df = collapse_depth_profile(
        radial_model_df.loc[radial_model_df["depth_km"] >= reference_crust_thickness_km], keep="last"
    )

    geometry_grid = xr.Dataset(
        {
            "surface_radius_km": xr.full_like(grail_crust_km, RMOON_KM).rename("surface_radius_km"),
            "crust_thickness_km": grail_crust_km,
            "moho_radius_km": (RMOON_KM - grail_crust_km).rename("moho_radius_km"),
        }
    )

    def sample_geometry(lat_deg, lon_deg):
        crust_thickness_km = sample_crust_thickness(grail_crust_km, lat_deg, lon_deg)
        return {
            "lat_deg": float(lat_deg),
            "lon_deg": float(lon_deg),
            "surface_radius_km": RMOON_KM,
            "crust_thickness_km": crust_thickness_km,
            "moho_radius_km": RMOON_KM - crust_thickness_km,
            "moho_depth_km": crust_thickness_km,
        }

    def sample_reference_profile(lat_deg, lon_deg, depth_grid_km):
        geometry = sample_geometry(lat_deg, lon_deg)
        depth_grid_km = np.asarray(depth_grid_km, dtype=float)

        if np.any(depth_grid_km < 0.0):
            raise ValueError("Depths must be non-negative and measured downward from the local surface.")

        local_crust_thickness_km = max(geometry["crust_thickness_km"], 1e-6)
        crust_mask = depth_grid_km <= local_crust_thickness_km

        reference_crust_depth_km = np.clip(
            depth_grid_km / local_crust_thickness_km * reference_crust_thickness_km,
            0.0,
            reference_crust_thickness_km,
        )
        reference_mantle_depth_km = reference_crust_thickness_km + np.clip(
            depth_grid_km - local_crust_thickness_km,
            0.0,
            None,
        )

        vp = np.empty_like(depth_grid_km)
        vs = np.empty_like(depth_grid_km)
        rho = np.empty_like(depth_grid_km)

        vp[crust_mask] = interpolate_profile(reference_crust_depth_km[crust_mask], crust_profile_df, "vp_km_s")
        vs[crust_mask] = interpolate_profile(reference_crust_depth_km[crust_mask], crust_profile_df, "vs_km_s")
        rho[crust_mask] = interpolate_profile(reference_crust_depth_km[crust_mask], crust_profile_df, "rho_g_cm3")

        vp[~crust_mask] = interpolate_profile(reference_mantle_depth_km[~crust_mask], mantle_profile_df, "vp_km_s")
        vs[~crust_mask] = interpolate_profile(reference_mantle_depth_km[~crust_mask], mantle_profile_df, "vs_km_s")
        rho[~crust_mask] = interpolate_profile(reference_mantle_depth_km[~crust_mask], mantle_profile_df, "rho_g_cm3")

        return pd.DataFrame(
            {
                "lat_deg": float(lat_deg),
                "lon_deg": float(lon_deg),
                "depth_from_surface_km": depth_grid_km,
                "radius_km": RMOON_KM - depth_grid_km,
                "distance_below_moho_km": np.maximum(depth_grid_km - geometry["crust_thickness_km"], 0.0),
                "region": np.where(crust_mask, "crust", "mantle_or_deeper"),
                "vp_km_s": vp,
                "vs_km_s": vs,
                "rho_g_cm3": rho,
                "surface_radius_km": RMOON_KM,
                "crust_thickness_km": geometry["crust_thickness_km"],
                "moho_radius_km": geometry["moho_radius_km"],
                "reference_crust_depth_km": reference_crust_depth_km,
                "reference_mantle_depth_km": reference_mantle_depth_km,
            }
        )

    return {
        "name": "3D_Reference_Model_Margarit",
        "geometry_grid": geometry_grid,
        "reference_profile_1d": radial_model_df,
        "reference_crust_thickness_km": reference_crust_thickness_km,
        "crust_profile_1d": crust_profile_df,
        "mantle_profile_1d": mantle_profile_df,
        "sample_geometry": sample_geometry,
        "sample_reference_profile": sample_reference_profile,
    }


def load_excel_long(xlsx, sheet, *, FC, BANDS):
    d = pd.read_excel(xlsx, sheet_name=sheet)

    need = ["starttime", "station", "fc_hz", "t0_dt_mean"]
    missing = [c for c in need if c not in d.columns]
    if missing:
        raise KeyError(f"Missing columns in sheet '{sheet}': {missing}; available: {list(d.columns)}")

    d["station"] = d["station"].astype(str)
    d["fc_hz"] = pd.to_numeric(d["fc_hz"], errors="coerce").astype(float)
    d["starttime_dt"] = pd.to_datetime(d["starttime"], errors="coerce", utc=True)
    d["t0_dt_mean_dt"] = pd.to_datetime(d["t0_dt_mean"], errors="coerce", utc=True)

    if "distance" in d.columns:
        d["distance_deg"] = pd.to_numeric(d["distance"], errors="coerce")
    elif "epi_deg" in d.columns:
        d["distance_deg"] = pd.to_numeric(d["epi_deg"], errors="coerce")
    else:
        d["distance_deg"] = np.nan

    lat_col = _first_existing_col(d.columns, ["lat", "Lat", "LAT", "latitude", "Latitude"])
    lon_col = _first_existing_col(d.columns, ["Lon", "lon", "LON", "longitude", "Longitude"])

    if lat_col is not None:
        d["lat"] = pd.to_numeric(d[lat_col], errors="coerce")
    else:
        d["lat"] = np.nan

    if lon_col is not None:
        d["Lon"] = pd.to_numeric(d[lon_col], errors="coerce")
    else:
        d["Lon"] = np.nan

    d = d[d["fc_hz"].isin(BANDS)].copy()
    d["event"] = d["starttime_dt"].astype(str) + "__" + d["station"]

    ref = (
        d[d["fc_hz"].eq(FC)][["event", "t0_dt_mean_dt"]]
        .rename(columns={"t0_dt_mean_dt": "t0_fc_dt"})
        .groupby("event", as_index=False)["t0_fc_dt"]
        .min()
    )
    d = d.merge(ref, on="event", how="left")
    d["dt_rel"] = (d["t0_dt_mean_dt"] - d["t0_fc_dt"]).dt.total_seconds()
    d = d[d["dt_rel"].notna() & d["starttime_dt"].notna() & d["t0_dt_mean_dt"].notna()].copy()

    return d[[
        "event",
        "station",
        "starttime_dt",
        "fc_hz",
        "dt_rel",
        "distance_deg",
        "lat",
        "Lon",
        "t0_dt_mean_dt",
    ]].rename(columns={"fc_hz": "band"})


def build_event_band_matrix(df_long, *, BANDS):
    return (
        df_long.pivot_table(index="event", columns="band", values="dt_rel", aggfunc="first")
        .reindex(columns=BANDS)
        .sort_index()
    )


def select_events(*, dt_mat, FC, BANDS, MIN_POST, K_NEG, K_PRE_POS=0, LOWER_TOL=0.0):
    post_bands = [b for b in BANDS if b > FC]
    pre_bands = [b for b in BANDS if b < FC]

    keep = []
    for ev in dt_mat.index:
        dt = dt_mat.loc[ev]

        post_vals = dt[post_bands].dropna()
        if len(post_vals) < MIN_POST:
            keep.append(False)
            continue

        if int((post_vals <= LOWER_TOL).sum()) > K_NEG:
            keep.append(False)
            continue

        pre_vals = dt[pre_bands].dropna()
        if int((pre_vals > LOWER_TOL).sum()) > K_PRE_POS:
            keep.append(False)
            continue

        keep.append(True)

    return pd.Series(keep, index=dt_mat.index, name="keep")


def build_selected_events_from_excel():
    df_long = load_excel_long(XLSX, SHEET, FC=FC, BANDS=BANDS)
    dt_mat = build_event_band_matrix(df_long, BANDS=BANDS)
    keep_mask = select_events(dt_mat=dt_mat, FC=FC, BANDS=BANDS, **SCENARIO)

    dist_map = df_long[["event", "distance_deg"]].drop_duplicates(subset=["event"]).set_index("event")["distance_deg"].to_dict()
    lat_map = df_long[["event", "lat"]].drop_duplicates(subset=["event"]).set_index("event")["lat"].to_dict()
    lon_map = df_long[["event", "Lon"]].drop_duplicates(subset=["event"]).set_index("event")["Lon"].to_dict()
    time_map = (
        df_long[df_long["band"] == FC][["event", "t0_dt_mean_dt"]]
        .dropna()
        .drop_duplicates(subset=["event"])
        .set_index("event")["t0_dt_mean_dt"]
        .to_dict()
    )

    rows = []
    for ev in keep_mask.index[keep_mask]:
        rows.append(
            dict(
                event=ev,
                time_utc=time_map.get(ev, pd.NaT),
                station=ev.split("__", 1)[-1],
                epi_deg=dist_map.get(ev, np.nan),
                lat=lat_map.get(ev, np.nan),
                Lon=lon_map.get(ev, np.nan),
            )
        )

    return pd.DataFrame(rows)


def load_event_catalog():
    selected_df = build_selected_events_from_excel()

    if SELECTED_EVENTS_CSV.exists():
        coords_df = pd.read_csv(SELECTED_EVENTS_CSV)
        selected_df = selected_df.drop(columns=[c for c in ["lat", "Lon"] if c in selected_df.columns])
        merge_cols = ["event", "lat", "Lon"]
        if "depth_km" in coords_df.columns:
            merge_cols.append("depth_km")
        df = selected_df.merge(coords_df[merge_cols].drop_duplicates(subset=["event"]), on="event", how="left")
    else:
        df = selected_df.copy()

    df["station"] = df["station"].astype(str).str.strip()
    for col in ["epi_deg", "lat", "Lon"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    if "depth_km" in df.columns:
        df["depth_km"] = pd.to_numeric(df["depth_km"], errors="coerce")
    else:
        df["depth_km"] = SOURCE_DEPTH_KM

    n_before_coords = len(df)
    df = df[
        np.isfinite(df["epi_deg"]) &
        np.isfinite(df["lat"]) &
        np.isfinite(df["Lon"]) &
        df["station"].isin(APOLLO_STATIONS.keys())
    ].copy()
    df["depth_km"] = df["depth_km"].fillna(SOURCE_DEPTH_KM)

    df = df.sort_values(["epi_deg", "time_utc"], na_position="last").drop_duplicates(subset=["event"])
    print(f"Scenario-selected events before coordinate filtering: {n_before_coords}")
    print(f"Scenario-selected events with usable coordinates: {len(df)}")
    if MAX_EVENTS is not None:
        df = df.head(MAX_EVENTS)
        print(f"Events kept after MAX_EVENTS filter: {len(df)}")

    return df.reset_index(drop=True)


In [5]:
grail_crust_km = load_grail_crust(GRAIL_GRID)
radial_model_df = read_vpremoon_model(MODEL_FILE)
Reference_Model_Margarit_3D = build_reference_model_margarit_3d(grail_crust_km, radial_model_df)

print(f"Model name: {Reference_Model_Margarit_3D['name']}")
print(f"Constant surface radius used in solver: {RMOON_KM:.1f} km")
print(f"Reference crust thickness from 1D template: {Reference_Model_Margarit_3D['reference_crust_thickness_km']:.1f} km")
print(
    "Kim crust min/mean/max (km): "
    f"{float(grail_crust_km.min()):.2f}, {float(grail_crust_km.mean()):.2f}, {float(grail_crust_km.max()):.2f}"
)

preview_profile_df = Reference_Model_Margarit_3D["sample_reference_profile"](
    lat_deg=0.0,
    lon_deg=3.6,
    depth_grid_km=np.array([0.0, 5.0, 10.0, 20.0, 35.0, 60.0, 100.0, 300.0]),
)
display(preview_profile_df.round(3))


Model name: 3D_Reference_Model_Margarit
Constant surface radius used in solver: 1737.4 km
Reference crust thickness from 1D template: 28.0 km
Kim crust min/mean/max (km): 6.06, 34.75, 62.61


ModuleNotFoundError: No module named 'scipy'

In [ ]:
def latlon_to_spherical(lat_deg, lon_deg, depth_km=0.0, radius_km=RMOON_KM):
    return np.array([
        radius_km - depth_km,
        np.deg2rad(90.0 - lat_deg),
        np.deg2rad(lon_deg % 360.0),
    ], dtype=float)


def spherical_to_cartesian(points):
    pts = np.asarray(points, dtype=float)
    r = pts[..., 0]
    theta = pts[..., 1]
    phi = pts[..., 2]
    sin_theta = np.sin(theta)
    x = r * sin_theta * np.cos(phi)
    y = r * sin_theta * np.sin(phi)
    z = r * np.cos(theta)
    return np.stack([x, y, z], axis=-1)


def nearest_node_index(coords, min_coords, node_intervals, npts):
    idx = np.rint((coords - min_coords) / node_intervals).astype(int)
    idx = np.clip(idx, 0, np.asarray(npts, dtype=int) - 1)
    return tuple(idx.tolist())


def build_velocity_grid(reference_model):
    mantle_profile_df = reference_model["mantle_profile_1d"]
    radius_min_km = max(1.0, RMOON_KM - float(mantle_profile_df["depth_km"].max()))

    radii = np.linspace(radius_min_km, RMOON_KM, NR)
    theta = np.linspace(0.0, np.pi, NTHETA)
    phi = np.linspace(0.0, 2.0 * np.pi, NPHI)

    lat_nodes = 90.0 - np.rad2deg(theta)
    lon_nodes = np.rad2deg(phi) % 360.0

    crust_samples = np.empty((NTHETA, NPHI), dtype=float)
    for i, lat_deg in enumerate(lat_nodes):
        for j, lon_deg in enumerate(lon_nodes):
            crust_samples[i, j] = reference_model["sample_geometry"](float(lat_deg), float(lon_deg))["crust_thickness_km"]

    local_depth_km = RMOON_KM - radii[:, None, None]
    local_crust_thickness_km = np.maximum(crust_samples[None, :, :], 1e-6)
    crust_mask = local_depth_km <= local_crust_thickness_km

    reference_crust_depth_km = np.clip(
        local_depth_km / local_crust_thickness_km * reference_model["reference_crust_thickness_km"],
        0.0,
        reference_model["reference_crust_thickness_km"],
    )
    reference_mantle_depth_km = reference_model["reference_crust_thickness_km"] + np.clip(
        local_depth_km - local_crust_thickness_km,
        0.0,
        None,
    )

    vp_crust = np.interp(
        reference_crust_depth_km.ravel(),
        reference_model["crust_profile_1d"]["depth_km"].to_numpy(dtype=float),
        reference_model["crust_profile_1d"]["vp_km_s"].to_numpy(dtype=float),
    ).reshape(reference_crust_depth_km.shape)
    vp_mantle = np.interp(
        reference_mantle_depth_km.ravel(),
        reference_model["mantle_profile_1d"]["depth_km"].to_numpy(dtype=float),
        reference_model["mantle_profile_1d"]["vp_km_s"].to_numpy(dtype=float),
    ).reshape(reference_mantle_depth_km.shape)

    velocity = np.where(crust_mask, vp_crust, vp_mantle)
    return velocity.astype(float), radius_min_km


def trace_ray_with_pykonal(solver, end_coords, *, min_points=2, min_arc_length_km=1.0):
    ray = np.asarray(solver.trace_ray(np.asarray(end_coords, dtype=float)), dtype=float)

    if ray.ndim != 2 or ray.shape[1] != 3:
        raise ValueError(f"Native pykonal trace_ray returned invalid shape: {ray.shape}")
    if len(ray) < min_points:
        raise ValueError(f"Native pykonal trace_ray returned too few samples: {len(ray)}")
    if not np.all(np.isfinite(ray)):
        raise ValueError("Native pykonal trace_ray returned non-finite values")

    segment_lengths = np.linalg.norm(np.diff(ray, axis=0), axis=1)
    if segment_lengths.sum() < min_arc_length_km:
        raise ValueError(
            "Native pykonal trace_ray returned a degenerate path "
            f"(total length={segment_lengths.sum():.6f} km)"
        )
    return ray


def trace_ray_from_gradient(field, end_coords, src_coords, step_km=TRACE_STEP_KM, max_steps=4000):
    coords = np.asarray(end_coords, dtype=float).copy()
    src = np.asarray(src_coords, dtype=float)
    path = [coords.copy()]
    value = np.inf

    for _ in range(max_steps):
        previous_value = value
        value = field.value(coords)
        if value >= previous_value or np.isnan(value):
            path.pop()
            break
        if np.linalg.norm(coords - src) <= 2.0 * step_km:
            path.append(src.copy())
            break

        grad = np.asarray(field.gradient.value(coords), dtype=float)
        grad_norm = np.linalg.norm(grad)
        if not np.isfinite(grad_norm) or grad_norm < 1e-12:
            break

        grad /= grad_norm
        grad[1] /= coords[0]
        grad[2] /= max(coords[0] * np.sin(coords[1]), 1e-6)

        new_coords = coords - step_km * grad
        new_coords[2] = new_coords[2] % (2.0 * np.pi)
        if np.linalg.norm(new_coords - coords) < 1e-8:
            break

        coords = new_coords
        path.append(coords.copy())

    return np.vstack(path)


def solve_event_with_pykonal(row, velocity, radius_min_km):
    if pykonal is None:
        raise ModuleNotFoundError("`pykonal` is required to solve the travel times in this notebook.")

    dr = (RMOON_KM - radius_min_km) / (NR - 1)
    dtheta = np.pi / (NTHETA - 1)
    dphi = 2.0 * np.pi / (NPHI - 1)

    solver = pykonal.EikonalSolver(coord_sys="spherical")
    solver.velocity.min_coords = (radius_min_km, 0.0, 0.0)
    solver.velocity.node_intervals = (dr, dtheta, dphi)
    solver.velocity.npts = (NR, NTHETA, NPHI)
    solver.velocity.values = velocity

    source_depth_km = float(row.get("depth_km", SOURCE_DEPTH_KM))
    src_coords = latlon_to_spherical(float(row["lat"]), float(row["Lon"]), source_depth_km)
    station = APOLLO_STATIONS[row["station"]]
    rec_coords = latlon_to_spherical(station["lat"], station["lon"], RECEIVER_DEPTH_KM)

    min_coords = np.asarray(solver.velocity.min_coords, dtype=float)
    node_intervals = np.asarray(solver.velocity.node_intervals, dtype=float)

    src_idx = nearest_node_index(src_coords, min_coords, node_intervals, solver.velocity.npts)
    solver.traveltime.values[src_idx] = 0.0
    solver.unknown[src_idx] = False
    solver.trial.push(*src_idx)
    solver.solve()

    rec_idx = nearest_node_index(rec_coords, min_coords, node_intervals, solver.velocity.npts)
    travel_time_s = float(solver.traveltime.values[rec_idx])

    trace_method = TRACE_METHOD
    if TRACE_METHOD == "native":
        try:
            ray_rtp = trace_ray_with_pykonal(solver, rec_coords)
        except Exception as exc:
            if STRICT_NATIVE_TRACE:
                raise
            print(f"Native pykonal trace_ray failed for {row['event']}: {exc}")
            ray_rtp = trace_ray_from_gradient(solver.traveltime, rec_coords, src_coords)
            trace_method = "gradient_fallback"
    elif TRACE_METHOD == "gradient":
        ray_rtp = trace_ray_from_gradient(solver.traveltime, rec_coords, src_coords)
    else:
        raise ValueError(f"Unsupported TRACE_METHOD: {TRACE_METHOD}")

    return {
        "event": row["event"],
        "station": row["station"],
        "epi_deg": float(row["epi_deg"]),
        "src_lat": float(row["lat"]),
        "src_lon": float(row["Lon"]),
        "src_depth_km": source_depth_km,
        "travel_time_s": travel_time_s,
        "ray_rtp": ray_rtp,
        "ray_xyz": spherical_to_cartesian(ray_rtp),
        "receiver_rtp": rec_coords,
        "trace_method": trace_method,
    }


In [ ]:
catalog_df = load_event_catalog()
velocity_3d, radius_min_km = build_velocity_grid(Reference_Model_Margarit_3D)

print(f"Events loaded: {len(catalog_df)}")
print(f"Velocity grid shape: {velocity_3d.shape}")
print(f"Velocity min/max (km/s): {float(np.nanmin(velocity_3d)):.3f}, {float(np.nanmax(velocity_3d)):.3f}")
display(catalog_df[["event", "station", "epi_deg", "lat", "Lon", "depth_km"]])


In [ ]:
solutions = []
for row in catalog_df.to_dict(orient="records"):
    solutions.append(solve_event_with_pykonal(row, velocity_3d, radius_min_km))

results_df = pd.DataFrame([
    {
        "event": sol["event"],
        "station": sol["station"],
        "epi_deg": sol["epi_deg"],
        "src_lat": sol["src_lat"],
        "src_lon": sol["src_lon"],
        "src_depth_km": sol["src_depth_km"],
        "travel_time_s": sol["travel_time_s"],
        "phase": PHASE_NAME,
        "trace_method": sol["trace_method"],
    }
    for sol in solutions
]).sort_values(["epi_deg", "station"]).reset_index(drop=True)

RESULTS_CSV.parent.mkdir(parents=True, exist_ok=True)
results_df.to_csv(RESULTS_CSV, index=False)

display(results_df)
print(f"Saved results to: {RESULTS_CSV}")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5), dpi=140)
colors = plt.cm.viridis(np.linspace(0.0, 1.0, max(len(solutions), 1)))
theta = np.linspace(0.0, 2.0 * np.pi, 400)

for ax, pair in zip(axes[:2], [(0, 2, "X", "Z"), (1, 2, "Y", "Z")]):
    ax.plot(RMOON_KM * np.cos(theta), RMOON_KM * np.sin(theta), color="0.7", lw=1.2)

    for color, sol in zip(colors, solutions):
        ray_xyz = sol["ray_xyz"]
        ax.plot(ray_xyz[:, pair[0]], ray_xyz[:, pair[1]], color=color, lw=1.6)
        ax.scatter(ray_xyz[-1, pair[0]], ray_xyz[-1, pair[1]], color=color, s=20)

    for station_name, info in APOLLO_STATIONS.items():
        st_xyz = spherical_to_cartesian(latlon_to_spherical(info["lat"], info["lon"], 0.0))
        ax.scatter(st_xyz[pair[0]], st_xyz[pair[1]], marker="^", color="k", s=35)
        ax.text(st_xyz[pair[0]], st_xyz[pair[1]], f" {station_name}", fontsize=8)

    ax.set_aspect("equal")
    ax.set_xlabel(f"{pair[2]} (km)")
    ax.set_ylabel(f"{pair[3]} (km)")
    ax.grid(alpha=0.25)

axes[0].set_title("Ray paths in X-Z")
axes[1].set_title("Ray paths in Y-Z")

axes[2].scatter(results_df["epi_deg"], results_df["travel_time_s"], c=np.arange(len(results_df)), cmap="viridis", s=45)
for row in results_df.itertuples(index=False):
    axes[2].text(row.epi_deg + 0.05, row.travel_time_s, row.station, fontsize=8)

axes[2].set_xlabel("Epicentral distance (deg)")
axes[2].set_ylabel("Travel time (s)")
axes[2].set_title("P travel times")
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()
